In [1]:
import os
from tqdm import tqdm
import gzip
import random

# base directories for the data
base_dirs = ["E:/Antibody Sequences/", "E:/Genbank Sequences Processed/", "E:/uniref100_processed/"]

# get all .txt.gz files in each of the base directories
antibody_files = []
genbank_files = []
uniref_files = []
for base_dir in base_dirs:
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith(".txt.gz"):
                if base_dir == base_dirs[0]:
                    antibody_files.append(os.path.join(root, file))
                elif base_dir == base_dirs[1]:
                    genbank_files.append(os.path.join(root, file))
                elif base_dir == base_dirs[2]:
                    uniref_files.append(os.path.join(root, file))
            

# shuffle the files
random.shuffle(antibody_files)
random.shuffle(genbank_files)
random.shuffle(uniref_files)

special_tokens = ["<EOS>", "<DNA>", "<mRNA>", "<RNA>", "<rRNA>", "<tRNA>",
                  "<cRNA>", "<ss-RNA>", "<ss-DNA>", "<ds-mRNA>", "<ds-rRNA>",
                  "<ds-RNA>", "<ms-DNA>", "<ms-RNA>", "<ds-cRNA>", "<protein>", "<antibody>"]

In [2]:
for file_num in range(0, 10):
    # generate ~10 billion chars of random substring text
    num_to_sample = int(1e10)
    text = ""
    pbar = tqdm(total=num_to_sample)
    num_files_processed = 0
    while len(text) < num_to_sample:
        # choose a file from a random base directory in order of the base directories
        if random.random() < 0.33 and len(antibody_files) > 0:
            file = antibody_files.pop()
        elif random.random() < 0.5 and len(genbank_files) > 0:
            file = genbank_files.pop()
        elif len(uniref_files) > 0:
            file = uniref_files.pop()
        elif len(antibody_files) > 0:
            file = antibody_files.pop()
        elif len(uniref_files) > 0:
            file = uniref_files.pop()
        elif len(genbank_files) > 0:
            file = genbank_files.pop()
        else:
            break # if there are no more files, break out of the loop

        text_parts = []

        with gzip.open(file, "rt") as f:
            try:
                lines = f.read().split("<EOS>")
            except:
                continue # if there is an error reading the file, skip it

            # there are some huge files, so only sample 2048 of the lines in each file
            lines = random.sample(lines, min(len(lines), 2048))

            for temp in lines:
                # Split each temp string into chunks of 2000 characters
                chunks = [temp[i:i+2000] for i in range(0, len(temp), 2000)]

                for i, chunk in enumerate(chunks):
                    if i < len(chunks) - 1:
                        text_parts.append(chunk + "\n")
                    else:
                        text_parts.append(chunk)
                    pbar.update(len(text_parts[-1]))
                text_parts.append("<EOS>\n")
                pbar.update(len(text_parts[-1]))
                if sum(len(part) for part in text_parts) + len(text) >= num_to_sample:
                    break
        num_files_processed += 1
        pbar.set_description(f"Creating file number {file_num + 1}... Processed {num_files_processed} files")

        text += ''.join(text_parts)
        pbar.update(len(text) - pbar.n)

    pbar.close()
    # save text to file
    with open(f"corpus_sample_{file_num}.txt", "w") as f:
        f.write(text)

Creating file number 1... Processed 168 files: : 10003563831it [06:08, 27143563.71it/s]                               
Creating file number 2... Processed 95 files: : 10420260281it [06:31, 26613337.79it/s]                               
Creating file number 3... Processed 114 files: : 10004594524it [07:53, 21150963.65it/s]                               
Creating file number 4... Processed 180 files: : 10506390692it [11:15, 15542413.77it/s]                               
Creating file number 5... Processed 157 files: : 10000376558it [11:47, 14134413.09it/s]                               
Creating file number 6... Processed 139 files: : 11868159126it [05:03, 39141266.73it/s]                               
Creating file number 7... Processed 123 files: : 10003853238it [04:35, 36334001.70it/s]                               
Creating file number 8... Processed 128 files: : 10020511989it [06:00, 27761880.87it/s]                               
Creating file number 9... Processed 159 files: : 

In [ ]:
# Training command
# ./spm_train --input=corpus_sample_0.txt,corpus_sample_1.txt,corpus_sample_2.txt,corpus_sample_3.txt,corpus_sample_4.txt,corpus_sample_5.txt,corpus_sample_6.txt,corpus_sample_7.txt,corpus_sample_8.txt,corpus_sample_9.txt --model_prefix=m --vocab_size=65536 --character_coverage=1.0 --user_defined_symbols="<EOS>,<DNA>,<mRNA>,<RNA>,<rRNA>,<tRNA>,<cRNA>,<ss-RNA>,<ss-DNA>,<ds-mRNA>,<ds-rRNA>,<ds-RNA>,<ms-DNA>,<ms-RNA>,<ds-cRNA>,<protein>,<antibody>" --input_sentence_size=50000 --shuffle_input_sentence=true --train_extremely_large_corpus=true --num_sub_iterations 10 --max_sentencepiece_length 32